In [0]:
%sql
-- fraud rate by amount band and period of the day / taxa de fraude por faixa de valor e período do dia
SELECT
  CASE
    WHEN amt < 50 THEN '1. até 50'
    WHEN amt < 250 THEN '2. 50 a 250'
    WHEN amt < 1000 THEN '3. 250 a 1000'
    ELSE '4. acima de 1000'
  END AS faixa_valor,
  CASE
    WHEN hour < 6 THEN 'madrugada'
    WHEN hour < 12 THEN 'manhã'
    WHEN hour < 18 THEN 'tarde'
    ELSE 'noite'
  END AS periodo,
  COUNT(*) AS transacoes,
  ROUND(AVG(is_fraud) * 100, 3) AS taxa_fraude_pct
FROM workspace.silver.transactions
GROUP BY 1, 2
ORDER BY 1, 2

%sql
-- monthly volume and fraud rate / volume e taxa de fraude por mês
SELECT
  DATE_TRUNC('month', event_time) AS mes,
  COUNT(*) AS transacoes,
  SUM(is_fraud) AS fraudes,
  ROUND(AVG(is_fraud) * 100, 3) AS taxa_fraude_pct
FROM workspace.silver.transactions
GROUP BY 1
ORDER BY 1

%sql
-- customers with the most frauds / clientes com mais fraudes
SELECT
  customer_hash,
  COUNT(*) AS transacoes,
  SUM(is_fraud) AS fraudes,
  ROUND(SUM(CASE WHEN is_fraud = 1 THEN amt ELSE 0 END), 2) AS valor_fraudado
FROM workspace.silver.transactions
GROUP BY customer_hash
HAVING SUM(is_fraud) > 0
ORDER BY fraudes DESC
LIMIT 20

"""Run a sql query and get a dataframe / roda uma query sql e recebe um dataframe."""

# query result as pandas for plotting / resultado da query em pandas para gráficos
df_sql = spark.sql("""
    SELECT category, ROUND(AVG(is_fraud) * 100, 3) AS taxa_fraude_pct
    FROM workspace.silver.transactions
    GROUP BY category
    ORDER BY taxa_fraude_pct DESC
""").toPandas()
df_sql